# Baseline: LGAR-style abstract screening with a decision model

Benchmark: the 20 CLEF TAR 2019 Intervention test reviews, with the eligibility criteria from LGAR.

- **BM25** (free): query = review title, as in LGAR.
- **Jev** (paid): the LGAR input (review title, research questions, inclusion and exclusion criteria, record title and abstract), asked as one yes/no question. The returned probability is the ranking score.

Metrics as in LGAR: MAP, recall@k%, WSS@95/100%, TNR@95%, macro-averaged over reviews.

Differences from LGAR: a yes/no probability instead of a 0-19 scale, and no monoT5 re-ranking (ties are broken by BM25 score).

Nothing paid runs unless `RUN_SMOKE` or `RUN_PAID` is set to `True` below.

Shared code (data, BM25, API calls, metrics) is in [src/screening.py](../../src/screening.py).

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False   # one paid call (a fraction of a cent) to check the API
RUN_PAID = False    # the full run; read the cost estimate first. Cached records are never re-requested.
OUT = S.out_dir("baseline")

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

Qrels from CLEF TAR, criteria from LGAR, titles and abstracts from PubMed. Everything is cached on disk.

In [2]:
records, CRITERIA = S.load_records()
records.groupby("review").agg(records=("pmid", "size"), incl_abs=("label_abs", "sum"), incl_content=("label_content", "sum"))

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


,records,incl_abs,incl_content
review,,,
CD000996,281,9,6
CD001261,571,72,22
CD004414,336,16,7
CD006468,3874,52,12
CD007867,943,17,9
CD009069,1757,78,5
CD009642,1922,62,50
CD010038,8867,23,9
CD010239,224,12,10


## BM25 (free)

In [3]:
records = S.add_bm25(records, CRITERIA)
results = {"BM25": S.evaluate(records, "bm25")}
results["BM25"]

,MAP,R@1%,R@5%,R@10%,R@20%,R@50%,WSS@95%,TNR@95%,WSS@100%,reviews
label_abs,22.3,12.2,34.4,46.2,60.7,82.6,33.3,39.5,28.7,20.0
label_content,17.0,12.0,30.2,47.0,65.2,86.7,43.6,49.5,44.6,18.0


## Decision model input

In [4]:
def build_body(review, title, abstract):
    c = CRITERIA[review]
    return {
        "model": S.MODEL,
        "state": {
            "review": {
                "title": c["title"],
                "research_questions": c["research_questions"],
                "inclusion_criteria": c["inclusion_criteria"],
                "exclusion_criteria": c["exclusion_criteria"],
            },
            "record": {"title": title, "abstract": abstract},
        },
        "questions": S.QUESTIONS,
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:1500])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "title": "Inhaled corticosteroids for bronchiectasis",
   "research_questions": "What is the efficacy and safety of inhaled corticosteroids (ICS) in children and adults with stable state bronchiectasis, specifically to assess whether the use of ICS: (1) reduces the severity and frequency of acute respiratory exacerbations; or (2) affects long-term pulmonary function decline?",
   "inclusion_criteria": "- Types of studies: all randomised controlled trials (RCTs) using inhaled corticosteroids (ICS) in patients with bronchiectasis, in comparison to placebo\n- Types of participants: children or adults with bronchiectasis (defined clinically or radiologically) not related to cystic fibrosis\n- Types of interventions: all types of ICS\n- Types of outcome measures: change in objective measures of lung function",
   "exclusion_criteria": "/"
  },
  "record": {
   "title": "Bronchiectasis Severity Is an Independent Risk Factor for Va

## Cost estimate

No API call. Tokens are approximated from characters, with the chars-per-token ratio measured in the smoke test (3.06).

In [5]:
S.estimate_cost(records, build_body)

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 3492 | max: 12784
estimated input tokens: 47.9M | estimated cost: $2.01


,records,chars,tokens_M,usd
review,,,,
CD000996,281,697021,0.228,0.010
CD001261,571,1487269,0.486,0.020
CD004414,336,1204609,0.394,0.017
CD006468,3874,10785905,3.525,0.148
CD007867,943,2945096,0.962,0.040
CD009069,1757,6606125,2.159,0.091
CD009642,1922,6555489,2.142,0.090
CD010038,8867,33922451,11.086,0.466
CD010239,224,708175,0.231,0.010


## Smoke test (one paid call)

In [6]:
if RUN_SMOKE:
    resp = S.call(build_body(*records.loc[0, ["review", "title", "abstract"]]))
    print(json.dumps(resp, indent=1))

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Records with a failed call or a refusal get the lowest score and are counted, not dropped.

In [8]:
records = S.add_scores(records, OUT, "jev_noul")
results["jev_noul"] = S.evaluate(records, "jev_noul")
pd.concat(results)

jev_noul: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


MAP  R@1%  R@5%  R@10%  R@20%  R@50%  WSS@95%  \
BM25     label_abs      22.3  12.2  34.4   46.2   60.7   82.6     33.3   
         label_content  17.0  12.0  30.2   47.0   65.2   86.7     43.6   
jev_noul label_abs      57.9  31.5  65.0   77.8   86.8   97.0     66.3   
         label_content  60.7  45.3  84.2   90.9   94.9   98.5     81.0   

                        TNR@95%  WSS@100%  reviews  
BM25     label_abs         39.5      28.7     20.0  
         label_content     49.5      44.6     18.0  
jev_noul label_abs         74.7      62.1     20.0  
         label_content     88.3      81.4     18.0

Reference from the LGAR paper on the same 20 reviews: LGAR MAP 56.8, fine-tuned BioBERT MAP 45.9. On all 31 TAR2019 test reviews: BM25 MAP 21.1, LGAR MAP 50.6, TNR@95% 76.5. LGAR's labels and record files may differ slightly from the ones built here.

In [9]:
THRESHOLD = 0.05
summary, per_review = S.threshold_table(records, "jev_noul", THRESHOLD)
print(f"threshold = {THRESHOLD}")
display(summary.T)
display(per_review)

threshold = 0.05


mean over reviews  all records pooled
label_abs     sensitivity               92.3                89.9
              specificity               75.6                77.7
              flagged                   27.8                23.8
label_content sensitivity               98.0                91.8
              specificity               74.0                77.1
              flagged                   27.8                23.8

label_abs                     label_content                    
         sensitivity specificity flagged   sensitivity specificity flagged
CD000996        88.9        82.4    19.9         100.0        81.8    19.9
CD001261        91.7        65.7    41.5          90.9        60.5    41.5
CD004414       100.0        88.1    16.1         100.0        85.7    16.1
CD006468        94.2        73.2    27.7         100.0        72.6    27.7
CD007867       100.0        51.6    49.3         100.0        51.2    49.3
CD009069        76.9        82.0    20.7         100.0        79.6    20.7
CD009642        98.4        83.7    19.0          98.0        83.1    19.0
CD010038       100.0        90.8     9.5         100.0        90.6     9.5
CD010239        91.7        95.3     9.4         100.0        94.9     9.4
CD010558        97.3        78.9    22.1         100.0        78.2    22.1
CD010753       100.0        42.8    57.7         100.0        42.6    57.7
CD011140       100.0        97.9     3.5           NaN        96.5     3.5
CD011571        93.3        89.3    19.2         100.0        84.3    19.2
CD011768       100.0        83.5    17.0         100.0        83.3    17.0
CD011977        89.8        76.0    40.5          89.2        70.9    40.5
CD012069        85.3        38.6    63.6          85.7        38.1    63.6
CD012164       100.0        42.6    62.3         100.0        39.7    62.3
CD012342        50.0        92.3     7.8           NaN        92.2     7.8
CD012455       100.0        97.7     2.8         100.0        97.5     2.8
CD012551        88.2        59.1    46.4         100.0        56.4    46.4